In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "s2_pore_phase.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "s2_pore_phase.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
from s2_pore_phase import s2_direction, s2_direction_torch, linear_autocorrelation_fft, linear_autocorrelation_fft_torch, overlap_counts, overlap_counts_torch, s2_radial_average, s2_radial_average_torch, style_axes, add_legend, turning_point_markevery, y_axis_label, curve_ylim, curve_yrange, resolve_device, backend_label

import argparse
import glob
import importlib.util
import os

import numpy as np

try:
    import torch
except ImportError:
    torch = None


# Pore phase definition:
# raw data is binary int8, where 0 = solid/skeleton and 1 = pore.
TESTB_FOLDER = r"../../data/real_binary"
DCB_FOLDER = r"../../data/baselines/dcgan"
STYLE_FOLDER = r"../../data/baselines/stylegan"
ST_FOLDER = r"../../data/baselines/stgan"
GEB_FOLDER = r"../../data/generated_binary"

DIMENSION = (128, 128, 128)  # interpreted as (Z, Y, X)
DATA_TYPE = np.int8
PORE_VALUE = 1
MAX_R = 150

OUTPUT_DIR = "S2_results"
DEVICE = "auto"
FIGURE_DPI = 600
INSET_START_R = 60
INSET_END_R = 120

CURVE_KEYS = {
    "x": "S2_x",
    "y": "S2_y",
    "z": "S2_z",
    "radial": "S2_radial",
}

GROUP_STYLES = {
    "TI": {
        "color": "#FF0000",
        "linewidth": 1.6,
        "linestyle": "-",
        "markersize": 5.5,
        "markerfacecolor": "white",
        "markeredgecolor": "#FF0000",
        "markeredgewidth": 1.8,
        "markevery": "turning_points",
        "alpha": 1.0,
        "zorder": 6,
    },
    "DCGAN": {
        "color": "#B700E5",
        "linewidth": 1.6,
        "linestyle": "-",
        "markersize": 7.2,
        "markerfacecolor": "#00E51A",
        "markeredgecolor": "#00E51A",
        "markeredgewidth": 1.2,
        "markevery": "turning_points",
        "alpha": 1.0,
        "zorder": 3,
    },
    "StyleGAN": {
        "color": "#9A9A9A",
        "linewidth": 1.6,
        "linestyle": "-",
        "markersize": 8.2,
        "markerfacecolor": "#9A9A9A",
        "markeredgecolor": "#9A9A9A",
        "markeredgewidth": 1.0,
        "markevery": "turning_points",
        "alpha": 0.75,
        "zorder": 2,
    },
    "STGAN": {
        "color": "#FFFB00",
        "linewidth": 1.4,
        "linestyle": "-",
        "markersize": 6.4,
        "markerfacecolor": "white",
        "markeredgecolor": "#FFFB00",
        "markeredgewidth": 1.8,
        "markevery": "turning_points",
        "alpha": 1.0,
        "zorder": 4,
    },
    "ours": {
        "color": "#0000FF",
        "linewidth": 2.0,
        "linestyle": "-",
        "marker": None,
        "solid_capstyle": "round",
        "alpha": 1.0,
        "zorder": 5,
    },
}

PLOT_RC = {
    "font.family": "Arial",
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "axes.linewidth": 1.1,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}


def load_pore_indicator(filepath, shape=DIMENSION, dtype=DATA_TYPE, pore_value=PORE_VALUE):
    raw = np.fromfile(filepath, dtype=dtype)
    expected_size = int(np.prod(shape))
    if raw.size != expected_size:
        raise ValueError(
            f"{filepath}: voxel count mismatch, got {raw.size}, expected {expected_size}. "
            f"Check DIMENSION or DATA_TYPE."
        )
    return (raw.reshape(shape) == pore_value).astype(np.float32)


















def calculate_one_file(filepath, max_r, do_radial=True, backend="numpy"):
    image = load_pore_indicator(filepath)

    if backend == "torch":
        device = torch.device("cuda")
        image_gpu = torch.as_tensor(image, dtype=torch.float32, device=device)
        sx = s2_direction_torch(image_gpu, axis=2, max_r=max_r)
        sy = s2_direction_torch(image_gpu, axis=1, max_r=max_r)
        sz = s2_direction_torch(image_gpu, axis=0, max_r=max_r)
        radial = s2_radial_average_torch(image_gpu, max_r=max_r) if do_radial else np.full(max_r + 1, np.nan)
        torch.cuda.empty_cache()
    else:
        sx = s2_direction(image, axis=2, max_r=max_r)
        sy = s2_direction(image, axis=1, max_r=max_r)
        sz = s2_direction(image, axis=0, max_r=max_r)
        radial = s2_radial_average(image, max_r=max_r) if do_radial else np.full(max_r + 1, np.nan)

    return {
        "file": os.path.basename(filepath),
        "porosity": float(image.mean()),
        "x": sx,
        "y": sy,
        "z": sz,
        "radial": radial,
    }


def save_figure(output_path, dpi=FIGURE_DPI):
    import matplotlib.pyplot as plt

    output_path = Path(output_path)
    plt.savefig(output_path, dpi=dpi)
    plt.savefig(output_path.with_suffix(".pdf"), dpi=dpi)






def draw_curve(ax, r, group_name, curves, key, label=True):
    style = GROUP_STYLES.get(group_name, {}).copy()
    if style.get("marker") is not None and style.get("markevery") == "turning_points":
        style["markevery"] = turning_point_markevery(r, curves[key])
    ax.plot(r, curves[key], label=group_name if label else None, **style)










def draw_plot_with_inset(output_path, group_curves, max_r, key, title, figsize=(10.0, 10.0)):
    import matplotlib.pyplot as plt

    with plt.rc_context(PLOT_RC):
        r = np.arange(max_r + 1)
        fig, ax = plt.subplots(figsize=figsize, dpi=600)

        for group_name, curves in group_curves.items():
            draw_curve(ax, r, group_name, curves, key)

        ax.set_xlabel("Lag distance r (voxel)")
        ax.set_ylabel(y_axis_label(key))
        style_axes(ax, 0, max_r)
        ax.set_ylim(*curve_yrange(group_curves, key, r, 0, max_r, pad=0.06))
        add_legend(ax, ncol=1)

        inset_start = min(INSET_START_R, max_r - 5)
        inset_end = min(INSET_END_R, max_r)
        if inset_end <= inset_start:
            inset_end = max_r
        inset = ax.inset_axes([0.23, 0.43, 0.56, 0.50])
        for group_name, curves in group_curves.items():
            draw_curve(inset, r, group_name, curves, key, label=False)
        style_axes(inset, inset_start, inset_end, hide_top_right=False)
        inset.set_ylim(*curve_yrange(group_curves, key, r, inset_start, inset_end, pad=0.12))
        inset.tick_params(labelsize=8, length=3)
        inset.tick_params(which="minor", length=1.8)
        zoom_indicator = ax.indicate_inset_zoom(inset, edgecolor="#777777", linewidth=0.8, alpha=0.75)
        zoom_artists = []
        if hasattr(zoom_indicator, "rectangle"):
            zoom_artists.append(zoom_indicator.rectangle)
        if hasattr(zoom_indicator, "connectors"):
            zoom_artists.extend(zoom_indicator.connectors)
        if not zoom_artists:
            for artist in zoom_indicator:
                if isinstance(artist, tuple):
                    zoom_artists.extend(artist)
                else:
                    zoom_artists.append(artist)
        for artist in zoom_artists:
            if artist is not None:
                artist.set_linestyle("--")
        fig.subplots_adjust(left=0.10, right=0.97, bottom=0.13, top=0.88)
    save_figure(output_path)
    plt.close(fig)


def save_radial_plot(output_path, group_curves, max_r):
    draw_plot_with_inset(
        output_path=output_path,
        group_curves=group_curves,
        max_r=max_r,
        key="radial",
        title="Pore-phase two-point correlation",
        figsize=(10.0, 10.0),
    )


def save_individual_curve_plots(output_dir, group_curves, max_r):
    for key, curve_label in CURVE_KEYS.items():
        if key == "radial" and any(np.all(np.isnan(curves[key])) for curves in group_curves.values()):
            continue
        draw_plot_with_inset(
            output_path=output_dir / f"{curve_label}_all_groups.png",
            group_curves=group_curves,
            max_r=max_r,
            key=key,
            title=curve_label,
            figsize=(10.0, 10.0),
        )


def calculate_group(group_name, folder, max_r, max_files=None, do_radial=True, backend="numpy"):
    files = sorted(glob.glob(os.path.join(folder, "*.raw")))
    if max_files is not None:
        files = files[:max_files]
    if not files:
        raise FileNotFoundError(f"No .raw files found in {folder}")

    print(f"\n[{group_name}] folder: {folder}")
    print(f"[{group_name}] files: {len(files)}")

    results = []
    for i, filepath in enumerate(files, start=1):
        print(f"  ({i}/{len(files)}) {os.path.basename(filepath)}")
        result = calculate_one_file(filepath, max_r=max_r, do_radial=do_radial, backend=backend)
        results.append(result)

    mean_curves = {
        "x": np.mean([res["x"] for res in results], axis=0),
        "y": np.mean([res["y"] for res in results], axis=0),
        "z": np.mean([res["z"] for res in results], axis=0),
        "radial": np.mean([res["radial"] for res in results], axis=0),
        "porosity_mean": float(np.mean([res["porosity"] for res in results])),
        "porosity_std": float(np.std([res["porosity"] for res in results], ddof=1)) if len(results) > 1 else 0.0,
        "file_count": len(results),
    }
    print(
        f"[{group_name}] mean porosity = {mean_curves['porosity_mean']:.6f}, "
        f"std = {mean_curves['porosity_std']:.6f}, phi^2 = {mean_curves['porosity_mean'] ** 2:.6f}"
    )
    return mean_curves


def parse_args():
    parser = argparse.ArgumentParser(description="Calculate pore-phase two-point correlation S2(r) for binary raw cores.")
    parser.add_argument("--testb-folder", default=TESTB_FOLDER, help="Folder of original TESTB .raw files.")
    parser.add_argument("--dcb-folder", default=DCB_FOLDER, help="Folder of DCB .raw files.")
    parser.add_argument("--style-folder", default=STYLE_FOLDER, help="Folder of STYLE .raw files.")
    parser.add_argument("--st-folder", default=ST_FOLDER, help="Folder of ST .raw files.")
    parser.add_argument("--geb-folder", default=GEB_FOLDER, help="Folder of GEB .raw files.")
    parser.add_argument("--max-r", type=int, default=MAX_R, help="Maximum lag distance in voxels.")
    parser.add_argument("--max-files", type=int, default=None, help="Optional limit for quick testing.")
    parser.add_argument("--no-radial", action="store_true", help="Skip FFT radial average and calculate only x/y/z directions.")
    parser.add_argument("--output-dir", default=OUTPUT_DIR, help="Output directory.")
    parser.add_argument(
        "--device",
        default=DEVICE,
        choices=("auto", "cpu", "gpu"),
        help="Compute device. 'auto' uses PyTorch GPU when available, otherwise CPU.",
    )
    args, _ = parser.parse_known_args()
    return args






def main():
    args = parse_args()
    args.max_r = min(args.max_r, min(DIMENSION) - 1)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    do_radial = not args.no_radial
    backend = resolve_device(args.device)
    print(f"Device: {backend_label(backend)}")

    groups = {
        "TI": args.testb_folder,
        "DCGAN": args.dcb_folder,
        "StyleGAN": args.style_folder,
        "STGAN": args.st_folder,
        "ours": args.geb_folder,
    }

    group_curves = {}
    for group_name, folder in groups.items():
        curves = calculate_group(
            group_name=group_name,
            folder=folder,
            max_r=args.max_r,
            max_files=args.max_files,
            do_radial=do_radial,
            backend=backend,
        )
        group_curves[group_name] = curves

    if do_radial:
        save_radial_plot(output_dir / "S2_radial_all_groups_with_phi2.png", group_curves, args.max_r)
    save_individual_curve_plots(output_dir, group_curves, args.max_r)

    print("\nDone.")
    print(f"Output directory: {output_dir.resolve()}")
    print("Saved plots only. No tabular files are written.")


if __name__ == "__main__":
    main()


In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "s2_pore_phase.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "s2_pore_phase.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
from s2_pore_phase import resolve_device, s2_direction, s2_direction_torch, linear_autocorrelation_fft, linear_autocorrelation_fft_torch, overlap_counts, overlap_counts_torch, s2_radial_average, s2_radial_average_torch, turning_point_markevery

import glob
import os

import numpy as np

try:
    import torch
except ImportError:
    torch = None
# "../../data/ablations/wo-dit-stage2-b"
# Defaults follow the first cell if present.
TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary")
DCB_FOLDER = globals().get("DCB_FOLDER", r"../../data/ablations/wo-dit-stage2-b")
STYLE_FOLDER = globals().get("STYLE_FOLDER", r"../../data/ablations/wo-stage2-b")
ST_FOLDER = globals().get("ST_FOLDER", r"../../data/ablations/wo-dit-b")
GEB_FOLDER = globals().get("GEB_FOLDER", r"../../data/generated_binary")

DIMENSION = globals().get("DIMENSION", (128, 128, 128))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)
PORE_VALUE = globals().get("PORE_VALUE", 1)
MAX_R = globals().get("MAX_R", 150)
OUTPUT_DIR = globals().get("OUTPUT_DIR", "S2_results")
DEVICE = globals().get("DEVICE", "auto")
FIGURE_DPI = globals().get("FIGURE_DPI", 600)
INSET_START_R = globals().get("INSET_START_R", 60)
INSET_END_R = globals().get("INSET_END_R", 120)

# Publication-style curve-plot defaults. These intentionally override any styles
# left in memory by the first cell, so rerunning this cell produces the same figure.
GROUP_ORDER = ["TI", "DCGAN", "StyleGAN", "STGAN", "ours"]
GROUP_LABELS = {
    "TI": "TI",
    "DCGAN": "DCGAN",
    "StyleGAN": "StyleGAN",
    "STGAN": "STGAN",
    "ours": "ours",
}
GROUP_STYLES = {
    "TI": {"color": "#000000", "linewidth": 1.2, "linestyle": "-", "alpha": 0.98, "zorder": 3},
    "DCGAN": {"color": "#A5A4A4", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 4},
    "STGAN": {"color": "#4656E7", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 5},
    "StyleGAN": {"color": "#19C0DD", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 6},
    "ours": {"color": "#01E71C", "linewidth": 1.5, "linestyle": "--", "alpha": 0.99, "zorder": 7},
}
GROUP_DASHES = {
    "DCGAN": (4.0, 2.4),
    "StyleGAN": (3.6, 2.0),
    "STGAN": (6.0, 2.4),
    "ours": (7.0, 2.0),
}

DIRECTION_LABELS = {
    "x": r"$x$ direction",
    "y": r"$y$ direction",
    "z": r"$z$ direction",
    "radial": "Radial average",
}

PLOT_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 10.6,
    "axes.labelsize": 12.6,
    "axes.titlesize": 12.0,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "#202020",
    "axes.labelcolor": "#181818",
    "xtick.labelsize": 10.4,
    "ytick.labelsize": 10.4,
    "xtick.color": "#202020",
    "ytick.color": "#202020",
    "legend.fontsize": 9.8,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "grid.color": "#DADADA",
    "grid.linewidth": 0.55,
    "grid.alpha": 0.62,
}

DISPLAY_TITLES = False
INSET_START_R = 0
INSET_END_R = 20
SMOOTH_WINDOW = 1  # 1 = no smoothing, keeps the curve as angular as the raw S2 points allow




def load_pore_indicator(filepath, shape=DIMENSION, dtype=DATA_TYPE, pore_value=PORE_VALUE):
    raw = np.fromfile(filepath, dtype=dtype)
    expected_size = int(np.prod(shape))
    if raw.size != expected_size:
        raise ValueError(
            f"{filepath}: voxel count mismatch, got {raw.size}, expected {expected_size}. "
            f"Check DIMENSION or DATA_TYPE."
        )
    return (raw.reshape(shape) == pore_value).astype(np.float32)


















def calculate_one_file(filepath, max_r, do_radial=True, backend="numpy"):
    image = load_pore_indicator(filepath)
    if backend == "torch":
        device = torch.device("cuda")
        image_gpu = torch.as_tensor(image, dtype=torch.float32, device=device)
        sx = s2_direction_torch(image_gpu, axis=2, max_r=max_r)
        sy = s2_direction_torch(image_gpu, axis=1, max_r=max_r)
        sz = s2_direction_torch(image_gpu, axis=0, max_r=max_r)
        radial = s2_radial_average_torch(image_gpu, max_r=max_r) if do_radial else np.full(max_r + 1, np.nan)
        torch.cuda.empty_cache()
    else:
        sx = s2_direction(image, axis=2, max_r=max_r)
        sy = s2_direction(image, axis=1, max_r=max_r)
        sz = s2_direction(image, axis=0, max_r=max_r)
        radial = s2_radial_average(image, max_r=max_r) if do_radial else np.full(max_r + 1, np.nan)
    return {
        "file": os.path.basename(filepath),
        "porosity": float(image.mean()),
        "x": sx,
        "y": sy,
        "z": sz,
        "radial": radial,
    }


def calculate_group(group_name, folder, max_r, max_files=None, do_radial=True, backend="numpy"):
    files = sorted(glob.glob(os.path.join(folder, "*.raw")))
    if max_files is not None:
        files = files[:max_files]
    if not files:
        raise FileNotFoundError(f"No .raw files found in {folder}")
    print(f"\n[{group_name}] folder: {folder}")
    print(f"[{group_name}] files: {len(files)}")
    results = []
    for i, filepath in enumerate(files, start=1):
        print(f"  ({i}/{len(files)}) {os.path.basename(filepath)}")
        result = calculate_one_file(filepath, max_r=max_r, do_radial=do_radial, backend=backend)
        results.append(result)
    mean_curves = {
        "x": np.mean([res["x"] for res in results], axis=0),
        "y": np.mean([res["y"] for res in results], axis=0),
        "z": np.mean([res["z"] for res in results], axis=0),
        "radial": np.mean([res["radial"] for res in results], axis=0),
        "porosity_mean": float(np.mean([res["porosity"] for res in results])),
        "porosity_std": float(np.std([res["porosity"] for res in results], ddof=1)) if len(results) > 1 else 0.0,
        "file_count": len(results),
    }
    print(
        f"[{group_name}] mean porosity = {mean_curves['porosity_mean']:.6f}, "
        f"std = {mean_curves['porosity_std']:.6f}, phi^2 = {mean_curves['porosity_mean'] ** 2:.6f}"
    )
    return mean_curves






def y_axis_label(key):
    labels = {
        "x": r"$S_2^{x}(r)$",
        "y": r"$S_2^{y}(r)$",
        "z": r"$S_2^{z}(r)$",
        "radial": r"$S_2(r)$",
    }
    return labels.get(key, r"$S_2(r)$")


def ordered_groups(group_curves):
    ordered = [name for name in GROUP_ORDER if name in group_curves]
    ordered.extend([name for name in group_curves if name not in ordered])
    return ordered


def smooth_curve(y, window=SMOOTH_WINDOW):
    y = np.asarray(y, dtype=float)
    if window <= 1 or y.size < 3:
        return y
    window = int(window)
    if window % 2 == 0:
        window += 1
    window = min(window, y.size if y.size % 2 == 1 else y.size - 1)
    if window < 3:
        return y
    kernel = np.ones(window, dtype=float) / window
    smoothed = y.copy()
    pad = window // 2
    padded = np.pad(smoothed, pad_width=pad, mode="edge")
    smoothed = np.convolve(padded, kernel, mode="valid")[: y.size]
    # Keep the physically important endpoints faithful when smoothing is enabled.
    smoothed[0] = y[0]
    smoothed[-1] = y[-1]
    return smoothed


def curve_yrange(group_curves, key, r, x_min, x_max, pad=0.10):
    r = np.asarray(r)
    mask = (r >= x_min) & (r <= x_max)
    values = []
    for curves in group_curves.values():
        curve = smooth_curve(np.asarray(curves[key], dtype=float)[: len(r)])
        values.append(curve[mask[: len(curve)]])
    values = np.concatenate(values)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    y_min = float(np.nanmin(values))
    y_max = float(np.nanmax(values))
    span = max(y_max - y_min, 1e-6)
    return max(0.0, y_min - span * pad), y_max + span * pad


def draw_curve(ax, r, group_name, curves, key, label=True):
    style = GROUP_STYLES.get(group_name, {}).copy()
    curve = smooth_curve(np.asarray(curves[key], dtype=float)[: len(r)])
    line, = ax.plot(
        r[: len(curve)],
        curve,
        label=GROUP_LABELS.get(group_name, group_name) if label else None,
        **style,
    )
    dash = GROUP_DASHES.get(group_name)
    if dash is not None:
        line.set_dashes(dash)
    if hasattr(line, "set_solid_capstyle"):
        line.set_solid_capstyle("butt")
    if hasattr(line, "set_dash_capstyle"):
        line.set_dash_capstyle("butt")
    if hasattr(line, "set_solid_joinstyle"):
        line.set_solid_joinstyle("miter")
    if hasattr(line, "set_dash_joinstyle"):
        line.set_dash_joinstyle("miter")
    return line


def style_axes(ax, x_min, x_max, hide_top_right=True, inset=False):
    from matplotlib.ticker import AutoMinorLocator, MaxNLocator

    ax.set_xlim(x_min, x_max)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=7 if not inset else 5, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6 if not inset else 5))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.grid(True, which="major", axis="both")
    ax.grid(False, which="minor")
    ax.tick_params(axis="both", which="major", direction="out", length=3.9, width=0.85, pad=4)
    ax.tick_params(axis="both", which="minor", direction="out", length=2.0, width=0.65)
    for spine in ax.spines.values():
        spine.set_color("#202020")
        spine.set_linewidth(1.0 if not inset else 0.9)
    ax.spines["top"].set_visible(not hide_top_right)
    ax.spines["right"].set_visible(not hide_top_right)


def add_legend(ax):
    legend = ax.legend(
        loc="upper right",
        frameon=False,
        handlelength=3.0,
        handletextpad=0.65,
        borderaxespad=0.2,
        labelspacing=0.42,
    )
    for line in legend.get_lines():
        line.set_linewidth(max(line.get_linewidth(), 2.5))
    return legend


def save_figure(output_path, dpi=None):
    import matplotlib.pyplot as plt

    if dpi is None:
        dpi = FIGURE_DPI
    output_path = Path(output_path)
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight", pad_inches=0.035)
    plt.savefig(output_path.with_suffix(".pdf"), dpi=dpi, bbox_inches="tight", pad_inches=0.035)
    plt.savefig(output_path.with_suffix(".svg"), dpi=dpi, bbox_inches="tight", pad_inches=0.035)


def draw_plot_with_inset(output_path, group_curves, max_r, key, title, figsize=(4, 4)):
    import matplotlib.pyplot as plt

    with plt.rc_context(PLOT_RC):
        groups = ordered_groups(group_curves)
        available_lengths = [len(np.asarray(group_curves[name][key])) for name in groups]
        plot_max_r = min(max_r, min(available_lengths) - 1)
        r = np.arange(plot_max_r + 1)

        fig, ax = plt.subplots(figsize=figsize, dpi=FIGURE_DPI)
        for group_name in groups:
            draw_curve(ax, r, group_name, group_curves[group_name], key)

        if DISPLAY_TITLES and title:
            ax.set_title(title, pad=7)
        ax.set_xlabel(r"Lag distance, $r$ (voxel)")
        ax.set_ylabel(y_axis_label(key))
        style_axes(ax, 0, plot_max_r)
        ax.set_ylim(*curve_yrange(group_curves, key, r, 0, plot_max_r, pad=0.075))
        ax.text(
            0.018,
            0.965,
            DIRECTION_LABELS.get(key, ""),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=11.2,
            color="#161616",
        )
        add_legend(ax)

        fig.subplots_adjust(left=0.105, right=0.985, bottom=0.12, top=0.965)
        save_figure(output_path)
        plt.close(fig)


def save_radial_plot(output_path, group_curves, max_r):
    draw_plot_with_inset(
        output_path=output_path,
        group_curves=group_curves,
        max_r=max_r,
        key="radial",
        title="Pore-phase two-point correlation",
        figsize=(4, 4),
    )


def save_individual_curve_plots(output_dir, group_curves, max_r):
    curve_keys = {"x": "S2_x", "y": "S2_y", "z": "S2_z", "radial": "S2_radial"}
    for key, curve_label in curve_keys.items():
        if key == "radial" and any(np.all(np.isnan(curves[key])) for curves in group_curves.values()):
            continue
        draw_plot_with_inset(
            output_path=output_dir / f"{curve_label}_all_groups.png",
            group_curves=group_curves,
            max_r=max_r,
            key=key,
            title=curve_label,
            figsize=(4, 4),
        )


output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
do_radial = True
backend = resolve_device(DEVICE)
print(f"Device: {backend}")

groups = {
    "TI": TESTB_FOLDER,
    "DCGAN": DCB_FOLDER,
    "STGAN": ST_FOLDER,
    "StyleGAN": STYLE_FOLDER,
    "ours": GEB_FOLDER,
}

group_curves = {}
for group_name, folder in groups.items():
    curves = calculate_group(
        group_name=group_name,
        folder=folder,
        max_r=MAX_R,
        max_files=None,
        do_radial=do_radial,
        backend=backend,
    )
    group_curves[group_name] = curves

if do_radial:
    save_radial_plot(output_dir / "S2_radial_all_groups_with_phi2.png", group_curves, MAX_R)
save_individual_curve_plots(output_dir, group_curves, MAX_R)

print("\nDone.")
print(f"Output directory: {output_dir.resolve()}")
print("Saved plots only. No tabular files are written.")


In [ ]:
# Standalone curve-area plots. This cell can run by itself.
import importlib.util
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Wedge


def load_s2_module():
    try:
        import s2_pore_phase as s2_module
        return s2_module
    except ModuleNotFoundError:
        pass

    candidate_dirs = [
        Path.cwd(),
        Path.cwd() / "evaluation/digital_core",
        Path.home() / "evaluation/digital_core",
        Path("evaluation/digital_core"),
    ]
    for candidate_dir in candidate_dirs:
        module_path = candidate_dir / "s2_pore_phase.py"
        if module_path.exists():
            sys.path.insert(0, str(candidate_dir))
            spec = importlib.util.spec_from_file_location("s2_pore_phase", module_path)
            s2_module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(s2_module)
            return s2_module

    raise ModuleNotFoundError("Cannot find s2_pore_phase.py. Put it in the notebook folder or add its folder to sys.path.")


s2 = load_s2_module()

TESTB_FOLDER = globals().get("TESTB_FOLDER", s2.TESTB_FOLDER)
DCB_FOLDER = globals().get("DCB_FOLDER", s2.DCB_FOLDER)
STYLE_FOLDER = globals().get("STYLE_FOLDER", s2.STYLE_FOLDER)
ST_FOLDER = globals().get("ST_FOLDER", s2.ST_FOLDER)
GEB_FOLDER = globals().get("GEB_FOLDER", s2.GEB_FOLDER)
DIMENSION = globals().get("DIMENSION", s2.DIMENSION)
DATA_TYPE = globals().get("DATA_TYPE", s2.DATA_TYPE)
PORE_VALUE = globals().get("PORE_VALUE", s2.PORE_VALUE)
MAX_R = globals().get("MAX_R", s2.MAX_R)
OUTPUT_DIR = globals().get("OUTPUT_DIR", s2.OUTPUT_DIR)
DEVICE = globals().get("DEVICE", s2.DEVICE)

s2.DIMENSION = DIMENSION
s2.DATA_TYPE = DATA_TYPE
s2.PORE_VALUE = PORE_VALUE

AREA_KEYS = {"x": "S2_x", "y": "S2_y", "z": "S2_z", "radial": "S2_radial"}
GROUP_ORDER = ["TI", "DCGAN", "StyleGAN", "STGAN", "ours"]
GROUP_LABELS = {"TI": "TI", "DCGAN": "DCGAN", "STGAN": "STGAN", "StyleGAN": "StyleGAN", "ours": "ours"}
GROUP_STYLES = {
    "TI": {"color": "#FF0000", "linewidth": 2.0, "linestyle": "--", "alpha": 0.98, "zorder": 7},
    "DCGAN": {"color": "#B8B8B8", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 3},
    "StyleGAN": {"color": "#8B0000", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 5},
    "STGAN": {"color": "#2936B4", "linewidth": 1.5, "linestyle": "--", "alpha": 0.96, "zorder": 4},
    "ours": {"color": "#00E51A", "linewidth": 2.0, "linestyle": "--", "alpha": 0.99, "zorder": 8},
}
GROUP_DASHES = {
    "TI": (8.0, 2.0),
    "DCGAN": (4.0, 2.4),
    "STGAN": (6.0, 2.4),
    "StyleGAN": (3.0, 1.8),
    "ours": (9.0, 2.0),
}


def ordered_groups(curves_by_group):
    ordered = [name for name in GROUP_ORDER if name in curves_by_group]
    ordered.extend([name for name in curves_by_group if name not in ordered])
    return ordered


def curve_area(curve, max_r=MAX_R):
    curve = np.asarray(curve, dtype=float)
    plot_max_r = min(int(max_r), curve.size - 1)
    r = np.arange(plot_max_r + 1)
    y = curve[: plot_max_r + 1]
    mask = np.isfinite(y)
    if mask.sum() < 2:
        return np.nan, np.nan
    r_valid = r[mask]
    y_valid = y[mask]
    integrate = getattr(np, "trapezoid", np.trapz)
    area = float(integrate(y_valid, r_valid))
    normalized_area = area / float(r_valid[-1] - r_valid[0]) if r_valid[-1] > r_valid[0] else np.nan
    return area, normalized_area


def calculate_area_group_curves():
    backend = s2.resolve_device(DEVICE)
    print(f"Device: {backend}")
    groups = {
        "TI": TESTB_FOLDER,
        "DCGAN": DCB_FOLDER,
        "StyleGAN": STYLE_FOLDER,
        "STGAN": ST_FOLDER,
        "ours": GEB_FOLDER,
    }
    calculated = {}
    for group_name, folder in groups.items():
        calculated[group_name] = s2.calculate_group(
            group_name=group_name,
            folder=folder,
            max_r=MAX_R,
            max_files=None,
            do_radial=True,
            backend=backend,
        )
    return calculated


if "group_curves" not in globals():
    group_curves = calculate_area_group_curves()
else:
    print("Using existing group_curves from this notebook session.")

area_rows = []
area_lookup = {}
for group_name in ordered_groups(group_curves):
    curves = group_curves[group_name]
    for key, label in AREA_KEYS.items():
        if key not in curves:
            continue
        area, normalized_area = curve_area(curves[key], MAX_R)
        area_rows.append((group_name, label, area, normalized_area))
        area_lookup[(group_name, key)] = area

print(f"Curve areas from r=0 to r={MAX_R}")
print(f"{'Group':<10} {'Curve':<10} {'Area':>14} {'Area/(r range)':>18}")
for group_name, label, area, normalized_area in area_rows:
    print(f"{group_name:<10} {label:<10} {area:14.6f} {normalized_area:18.6f}")

try:
    import pandas as pd

    area_table = pd.DataFrame(area_rows, columns=["group", "curve", "area", "normalized_area"])
    display(area_table)
except ImportError:
    area_table = area_rows

area_output_dir = Path(OUTPUT_DIR) / "area_circle_plots"
area_output_dir.mkdir(parents=True, exist_ok=True)

DIRECTION_AREA_KEYS = ["x", "y", "z"]


def direction_area(group_name, key):
    value = area_lookup.get((group_name, key), np.nan)
    return float(value) if np.isfinite(value) and value >= 0 else np.nan


def direction_fit_overlap_score(group_name, key):
    """Pointwise area overlap between a model S2 curve and TI: integral(min) / integral(max)."""
    if group_name not in group_curves or "TI" not in group_curves:
        return np.nan
    if key not in group_curves[group_name] or key not in group_curves["TI"]:
        return np.nan
    model_curve = np.asarray(group_curves[group_name][key], dtype=float)
    ti_curve = np.asarray(group_curves["TI"][key], dtype=float)
    plot_max_r = min(int(MAX_R), model_curve.size - 1, ti_curve.size - 1)
    if plot_max_r < 1:
        return np.nan
    r = np.arange(plot_max_r + 1)
    model_y = model_curve[: plot_max_r + 1]
    ti_y = ti_curve[: plot_max_r + 1]
    mask = np.isfinite(model_y) & np.isfinite(ti_y)
    if mask.sum() < 2:
        return np.nan
    r_valid = r[mask]
    model_valid = np.maximum(model_y[mask], 0.0)
    ti_valid = np.maximum(ti_y[mask], 0.0)
    integrate = getattr(np, "trapezoid", np.trapz)
    shared_area = float(integrate(np.minimum(model_valid, ti_valid), r_valid))
    union_area = float(integrate(np.maximum(model_valid, ti_valid), r_valid))
    return shared_area / union_area if union_area > 0 else np.nan


def direction_circle_radius(group_name, key):
    return 1.0


def equal_circle_overlap_fraction(distance):
    distance = float(np.clip(distance, 0.0, 2.0))
    if distance <= 0:
        return 1.0
    if distance >= 2:
        return 0.0
    return float((2.0 * np.arccos(distance / 2.0) - 0.5 * distance * np.sqrt(4.0 - distance ** 2)) / np.pi)


def circle_distance_for_overlap(target_overlap):
    if not np.isfinite(target_overlap):
        return 2.0
    target_overlap = float(np.clip(target_overlap, 0.0, 1.0))
    low, high = 0.0, 2.0
    for _ in range(48):
        mid = (low + high) / 2.0
        if equal_circle_overlap_fraction(mid) > target_overlap:
            low = mid
        else:
            high = mid
    return (low + high) / 2.0


def style_for_circle(group_name):
    style = GROUP_STYLES.get(group_name, {}).copy()
    return {
        "color": style.get("color", "#333333"),
        "linewidth": max(style.get("linewidth", 1.5), 2.1),
        "linestyle": style.get("linestyle", "--"),
        "alpha": 0.36 if group_name != "TI" else 0.24,
        "zorder": style.get("zorder", 3),
    }


def unequal_circle_overlap_area(r_big, r_small, distance):
    distance = float(abs(distance))
    if distance >= r_big + r_small:
        return 0.0
    if distance <= abs(r_big - r_small):
        return np.pi * min(r_big, r_small) ** 2
    big_term = r_big ** 2 * np.arccos((distance ** 2 + r_big ** 2 - r_small ** 2) / (2 * distance * r_big))
    small_term = r_small ** 2 * np.arccos((distance ** 2 + r_small ** 2 - r_big ** 2) / (2 * distance * r_small))
    lens_term = 0.5 * np.sqrt(
        max(0.0, (-distance + r_big + r_small) * (distance + r_big - r_small) * (distance - r_big + r_small) * (distance + r_big + r_small))
    )
    return float(big_term + small_term - lens_term)


def distance_for_small_circle_overlap(target_overlap, r_big, r_small):
    target_overlap = float(np.clip(target_overlap, 0.0, 1.0))
    full_small_area = np.pi * r_small ** 2
    low = max(0.0, r_big - r_small)
    high = r_big + r_small
    for _ in range(56):
        mid = (low + high) / 2.0
        frac = unequal_circle_overlap_area(r_big, r_small, mid) / full_small_area
        if frac > target_overlap:
            low = mid
        else:
            high = mid
    return (low + high) / 2.0


CIRCLE_TI_RADIUS = 0.72
CIRCLE_MODEL_RADIUS = 0.72
CIRCLE_LIMIT_REFERENCE_KEY = "z"


def circle_axis_limit(key):
    available_groups = [name for name in ordered_groups(group_curves) if np.isfinite(direction_area(name, key))]
    model_groups = [name for name in available_groups if name != "TI"]
    if not model_groups:
        return CIRCLE_TI_RADIUS * 1.32
    scores = {name: direction_fit_overlap_score(name, key) for name in model_groups}
    model_groups = sorted(model_groups, key=lambda name: scores[name])
    angle_positions = np.linspace(0, 2 * np.pi, max(len(model_groups), 1), endpoint=False)
    limit = CIRCLE_TI_RADIUS
    for draw_index, group_name in enumerate(model_groups):
        score = scores[group_name]
        if not np.isfinite(score):
            continue
        theta = angle_positions[draw_index]
        distance = circle_distance_for_overlap(score) * CIRCLE_TI_RADIUS
        center = np.array([np.cos(theta) * distance, np.sin(theta) * distance])
        limit = max(limit, abs(center[0]) + CIRCLE_MODEL_RADIUS, abs(center[1]) + CIRCLE_MODEL_RADIUS)
    return limit * 1.32


CIRCLE_AXIS_LIMIT = circle_axis_limit(CIRCLE_LIMIT_REFERENCE_KEY)


def draw_direction_circle_plot(key):
    ti_area = direction_area("TI", key)
    if not np.isfinite(ti_area) or ti_area <= 0:
        print(f"Skip {AREA_KEYS.get(key, key)}: TI area is not finite or <= 0.")
        return

    available_groups = [name for name in ordered_groups(group_curves) if np.isfinite(direction_area(name, key))]
    model_groups = [name for name in available_groups if name != "TI"]
    if not model_groups:
        return

    scores = {name: direction_fit_overlap_score(name, key) for name in model_groups}
    model_groups = sorted(model_groups, key=lambda name: scores[name])

    fig, ax = plt.subplots(figsize=(7.2, 7.2), dpi=600)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor("white")

    # Smaller circles than the previous overlapping version, with TI kept at the center.
    ti_radius = CIRCLE_TI_RADIUS
    model_radius = CIRCLE_MODEL_RADIUS
    angle_positions = np.linspace(0, 2 * np.pi, max(len(model_groups), 1), endpoint=False)
    circle_rows = []
    legend_handles = []

    for draw_index, group_name in enumerate(model_groups):
        score = scores[group_name]
        if not np.isfinite(score):
            continue
        theta = angle_positions[draw_index]
        distance = circle_distance_for_overlap(score) * ti_radius
        center = np.array([np.cos(theta) * distance, np.sin(theta) * distance])
        style = style_for_circle(group_name)
        color = style["color"]
        circle = Circle(
            center,
            model_radius,
            facecolor=color,
            edgecolor=color,
            linewidth=1.8,
            linestyle="--",
            alpha=0.28,
            zorder=style["zorder"],
            label=f"{GROUP_LABELS.get(group_name, group_name)}  overlap={score:.3f}",
        )
        ax.add_patch(circle)
        ax.add_patch(Circle(center, model_radius, facecolor="none", edgecolor=color, linewidth=1.9, linestyle="--", alpha=0.78, zorder=style["zorder"] + 10))
        legend_handles.append(circle)

        label_pos = center * 0.45
        ax.text(
            label_pos[0],
            label_pos[1],
            GROUP_LABELS.get(group_name, group_name),
            ha="center",
            va="center",
            fontsize=9.0,
            color="#222222",
            zorder=40,
        )
        circle_rows.append((center, model_radius))

    ti_circle = Circle(
        (0, 0),
        ti_radius,
        facecolor="none",
        edgecolor="#B00000",
        linewidth=2.6,
        linestyle="--",
        alpha=1.0,
        zorder=25,
        label="TI reference",
    )
    ax.add_patch(ti_circle)
    ax.add_patch(Circle((0, 0), ti_radius, facecolor="none", edgecolor="#B00000", linewidth=1.2, alpha=0.95, zorder=35))
    ax.text(0, 0, "TI", ha="center", va="center", fontsize=11, fontweight="bold", color="#B00000", zorder=45)
    legend_handles.append(ti_circle)

    limit = CIRCLE_AXIS_LIMIT
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title("Circular area overlap with TI", pad=14, fontsize=14)
    ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, -0.035), ncol=2, frameon=True, fontsize=8.5)
    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()
    circle_png = area_output_dir / f"{AREA_KEYS.get(key, key)}_area_circle_overlap_all_models.png"
    circle_pdf = area_output_dir / f"{AREA_KEYS.get(key, key)}_area_circle_overlap_all_models.pdf"
    fig.savefig(circle_png, bbox_inches="tight")
    fig.savefig(circle_pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved {AREA_KEYS.get(key, key)} circular overlap plot: {circle_png}")


fit_rows = []
for key in ["x", "y", "z"]:
    for group_name in ordered_groups(group_curves):
        if group_name == "TI":
            continue
        fit_rows.append((group_name, AREA_KEYS[key], direction_fit_overlap_score(group_name, key)))

print("\nCurve-fit area overlap relative to TI")
print(f"{'Group':<10} {'Curve':<10} {'Fit overlap':>14}")
for group_name, label, score in fit_rows:
    print(f"{group_name:<10} {label:<10} {score:14.6f}")

try:
    fit_table = pd.DataFrame(fit_rows, columns=["group", "curve", "fit_overlap"])
    display(fit_table)
except NameError:
    fit_table = fit_rows

for key in DIRECTION_AREA_KEYS:
    draw_direction_circle_plot(key)

print("Circle area plots: fit = integral(min(model, TI)) / integral(max(model, TI)); closer to 1 means the full S2 curve fits TI better.")
print(f"Saved circular overlap plots to: {area_output_dir.resolve()}")
